# Preprocessing: from two reference structures to DFT inputs

berry-flux-diag computes the change in polarization between a polar and a non-polar structure from their DFT wavefunctions. This notebook writes the DFT inputs for those runs; `postprocess_DFT_files.ipynb` then computes the polarization from them.

The example is BaTiO₃. To use your own material, change the structure files in step 1 of your section and the settings in **Setup**.

## Which calculation do you need?

Answer three questions, then go to the section in the table below. Each
section is complete on its own: run the **Setup** cells, then only your
section, top to bottom.

1. **Which DFT code?** VASP or Quantum ESPRESSO (QE).
2. **Is the material magnetic?**
   - **No:** use a *non-spin-polarized* calculation.
   - **Yes,** with collinear moments (ferromagnetic or antiferromagnetic,
     all moments along one axis): use a *spin-polarized* calculation
     (`ISPIN = 2` in VASP, `nspin = 2` in QE).
   - Non-collinear moments and spin-orbit coupling are **not supported**.
3. **Full Brillouin zone or irreducible Brillouin zone?**
   - **Full Brillouin zone:** the simplest choice. One DFT run per
     structure, with symmetry switched off.
   - **IBZ:** cheaper DFT, since only the symmetry-irreducible k-points are
     computed (BaTiO₃: 18 instead of 216 in VASP). Each structure needs two
     runs, an SCF and then an NSCF on a k-point list computed here.
     berry-flux-diag unfolds the states to the full zone when it reads them,
     and gives the same polarization to about 10⁻³ µC/cm².

<a id="contents"></a>
## Contents

| | full Brillouin zone | irreducible Brillouin zone (IBZ) |
|---|---|---|
| **VASP, non-spin-polarized** | [VASP, non-spin, FBZ](#vasp-nospin-fbz) | [VASP, non-spin, IBZ](#vasp-nospin-ibz) |
| **VASP, spin-polarized** | [VASP, spin, FBZ](#vasp-spin-fbz) | [VASP, spin, IBZ](#vasp-spin-ibz) |
| **Quantum ESPRESSO, non-spin-polarized** | [QE, non-spin, FBZ](#qe-nospin-fbz) | [QE, non-spin, IBZ](#qe-nospin-ibz) |
| **Quantum ESPRESSO, spin-polarized** | [QE, spin, FBZ](#qe-spin-fbz) | [QE, spin, IBZ](#qe-spin-ibz) |

## Setup

Run these cells first, whichever section you use.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by pymatgen

from pathlib import Path

import numpy as np
from pymatgen.core import Structure
from pymatgen.io.vasp import Poscar

import berry_flux_diag as bfd
from berry_flux_diag import symmetry

bfd.configure_logging("WARNING");  # "INFO" reports each step

OUT = Path("bfd_inputs")  # every section writes its inputs under here

Helper functions used by the sections below.

In [2]:
def image_tags(structs):
    """Names for the structures, in order: pol_orig, interp_1, ..., np_trans."""
    return ["pol_orig"] + [f"interp_{i}" for i in range(1, len(structs) - 1)] + ["np_trans"]


def shared_ibz_kpoints(structs, mesh, magmoms=None):
    """The k-point list for IBZ runs, and each point's weight.

    Reduced with only the symmetry operations each adjacent pair of
    structures shares (plus time reversal), because both runs of a pair must
    use one identical list. With interpolated images, every pair must give
    the same list, so that each structure is run once.
    """
    lists = []
    for a, b in zip(structs[:-1], structs[1:]):
        shared = symmetry.shared_symmetry(a, b, pol_magmoms=magmoms, np_magmoms=magmoms)
        reps, mult = symmetry.reduce_mesh(mesh, shared.k_rotations)
        print(f"{shared.space_groups['pol']} and {shared.space_groups['np']}: "
              f"{len(shared)} shared operations, {len(mesh)} -> {len(reps)} k-points")
        lists.append((reps, mult))
    for reps, mult in lists[1:]:
        if not (np.array_equal(reps, lists[0][0]) and np.array_equal(mult, lists[0][1])):
            raise ValueError("adjacent pairs share different symmetry, so no one IBZ list "
                             "serves the whole chain: use the full Brillouin zone instead")
    reps, mult = lists[0]
    return mesh[reps], mult


def write_incar(path, tags):
    Path(path).write_text("".join(f"{key} = {value}\n" for key, value in sorted(tags.items())))


def write_pw_input(path, structure, prefix, settings, *, calculation, kpoints_card,
                   use_symmetry=False, spin=None):
    """A pw.x input. spin, for a spin-polarized run, is a dict of &SYSTEM
    entries such as {"nspin": 2, "tot_magnetization": 0, ...}."""
    system = {"ibrav": 0, "ecutwfc": settings["ecutwfc"], "input_dft": "'pbe'",
              "nbnd": settings["nbnd"], "nat": len(structure),
              "ntyp": len(settings["species"]), **(spin or {})}
    if not use_symmetry:
        system.update(nosym=".TRUE.", noinv=".TRUE.")
    lines = ["&CONTROL", f"  calculation = '{calculation}'", f"  prefix = '{prefix}'",
             f"  pseudo_dir = '{settings['pseudo_dir']}'", "  outdir = 'calculations/'",
             "  verbosity = 'high'", "/", "&SYSTEM"]
    lines += [f"  {key} = {value}" for key, value in system.items()]
    lines += ["/", "&ELECTRONS", "  mixing_beta = 0.7", "  conv_thr = 1d-08",
              "  adaptive_thr = .TRUE.", "  mixing_mode = 'plain'", "  mixing_ndim = 8",
              "  diagonalization = 'david'", "/", "ATOMIC_SPECIES"]
    lines += [f"  {el}  {mass:.4f} {pseudo}" for el, mass, pseudo in settings["species"]]
    lines += ["ATOMIC_POSITIONS crystal"]
    lines += [f"  {site.specie.symbol} {f[0]:.6f} {f[1]:.6f} {f[2]:.6f}"
              for site, f in zip(structure, structure.frac_coords)]
    lines += [kpoints_card.rstrip("\n"), "CELL_PARAMETERS angstrom"]
    lines += [f"  {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}" for v in structure.lattice.matrix]
    Path(path).write_text("\n".join(lines) + "\n")

Your DFT settings. Edit these for your own material: everything except what berry-flux-diag needs, which each section adds.

In [3]:
# --- VASP: your usual static-calculation settings. -------------------------
# These are the BaTiO3 example's (PBEsol). Each section adds the tags that
# berry-flux-diag needs on top of them.
BASE_INCAR = {
    "ALGO": "Fast", "EDIFF": 1e-05, "ENAUG": 1360, "ENCUT": 680.0, "GGA": "Ps",
    "ISMEAR": -5, "LAECHG": True, "LASPH": True, "LCHARG": True, "LELF": False,
    "LMAXMIX": 4, "LMIXTAU": True, "LORBIT": 11, "LREAL": False, "LVTOT": True,
    "NELM": 200, "NSW": 0, "PREC": "Accurate", "SIGMA": 0.2,
}
VASP_MESH = (6, 6, 6)  # Monkhorst-Pack

# --- Quantum ESPRESSO: your usual settings. ----------------------------------
QE_SETTINGS = {
    "pseudo_dir": "/global/homes/p/pabigail/qe_pseudos/",  # where your pseudopotentials are
    "species": [("Ba", 137.3270, "Ba_ONCV_PBE-1.0.upf"),    # (element, mass, pseudopotential)
                ("O", 15.9994, "O_ONCV_PBE-1.0.upf"),
                ("Ti", 47.8670, "Ti_ONCV_PBE-1.0.upf")],
    "ecutwfc": 40,  # Ry
    "nbnd": 24,     # fixed, so every run has the same bands
}
QE_MESH, QE_SHIFT = (5, 5, 5), (1, 1, 1)  # K_POINTS automatic 5 5 5 1 1 1

MATERIAL = "BaTiO3"

<a id="vasp-nospin-fbz"></a>
## VASP: non-spin-polarized, full Brillouin zone

What you will set up: one VASP run per structure; no spin polarization; symmetry switched off, so the wavefunctions cover every k-point.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [4]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run: ['pol_orig', 'np_trans']


### 2. Write the VASP inputs

Every run needs:
- `ISYM = -1`: no symmetry, so every k-point of the mesh is computed;
- `LWAVE = .TRUE.`: write the `WAVECAR`;
- `NBANDS` fixed, so all runs have the same bands;
- the same `KPOINTS` mesh for every structure.

A `POTCAR` can't be generated here: add your own to each run directory, with the elements in the order of the `POSCAR`.

In [5]:
case = OUT / "VASP_nospin_fbz"
incar = {**BASE_INCAR, "ISYM": -1, "LWAVE": True, "NBANDS": 32,
         "ISPIN": 1}
kpoints = "Automatic mesh\n0\nMonkhorst\n" + " ".join(map(str, VASP_MESH)) + "\n"

for tag, structure in zip(tags, structs):
    run = case / tag
    run.mkdir(parents=True, exist_ok=True)
    Poscar(structure).write_file(run / "POSCAR")
    write_incar(run / "INCAR", incar)
    (run / "KPOINTS").write_text(kpoints)
print("wrote", sorted(str(p) for p in case.glob("*/*")))

wrote ['bfd_inputs/VASP_nospin_fbz/np_trans/INCAR', 'bfd_inputs/VASP_nospin_fbz/np_trans/KPOINTS', 'bfd_inputs/VASP_nospin_fbz/np_trans/POSCAR', 'bfd_inputs/VASP_nospin_fbz/pol_orig/INCAR', 'bfd_inputs/VASP_nospin_fbz/pol_orig/KPOINTS', 'bfd_inputs/VASP_nospin_fbz/pol_orig/POSCAR']


### 3. Run the calculations

In each structure's directory, with your `POTCAR` copied in:

```bash
vasp_std > vasp.out
```

Keep each run's `POSCAR`, `POTCAR`, `WAVECAR` and `vasprun.xml`.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#vasp-nospin-fbz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="vasp-nospin-ibz"></a>
## VASP: non-spin-polarized, irreducible Brillouin zone (IBZ)

What you will set up: two VASP runs per structure: an SCF with symmetry on, then an NSCF on the shared IBZ list; no spin polarization; berry-flux-diag unfolds the IBZ states to the full zone when it reads them.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [6]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run: ['pol_orig', 'np_trans']


### 2. The shared IBZ k-point list

Both runs of a pair must use one identical k-point list, reduced with only the symmetry operations the two structures share. Each code's own automatic reduction would give each structure a different list, so it is computed here. VASP's Monkhorst-Pack mesh is shifted by half a step along even axes.

In [7]:
full_mesh = symmetry.monkhorst_pack(VASP_MESH, shift=[1 if n % 2 == 0 else 0 for n in VASP_MESH])
ibz_kpoints, ibz_weights = shared_ibz_kpoints(structs, full_mesh)

P4mm (99) and Pm-3m (221): 8 shared operations, 216 -> 18 k-points


### 3. Write the VASP inputs

- **SCF:** symmetry on (no `ISYM`), on the usual mesh.
- **NSCF:** `ICHARG = 11` (read the SCF's charge density), `ISYM = 0` (keep the k-point list as given), the explicit IBZ `KPOINTS`, and `LWAVE = .TRUE.`.
- `NBANDS` is fixed, so the SCF and NSCF, and every structure, have the same bands.

A `POTCAR` can't be generated here: add your own to each run directory, with the elements in the order of the `POSCAR`.

In [8]:
case = OUT / "VASP_nospin_ibz"
scf_incar = {**BASE_INCAR, "LWAVE": True, "NBANDS": 32, "ISPIN": 1}
# The NSCF reads the SCF's CHGCAR and keeps the k-point list as given. The
# tetrahedron method (ISMEAR = -5) needs a generated mesh, so it uses
# Gaussian smearing instead.
nscf_incar = {**scf_incar, "ICHARG": 11, "ISYM": 0, "ISMEAR": 0, "SIGMA": 0.01,
              "LCHARG": False, "LAECHG": False, "LVTOT": False}
scf_kpoints = "Automatic mesh\n0\nMonkhorst\n" + " ".join(map(str, VASP_MESH)) + "\n"
nscf_kpoints = symmetry.vasp_kpoints_text(ibz_kpoints, ibz_weights)

for tag, structure in zip(tags, structs):
    for step, incar, kpoints in (("scf", scf_incar, scf_kpoints), ("nscf", nscf_incar, nscf_kpoints)):
        run = case / tag / step
        run.mkdir(parents=True, exist_ok=True)
        Poscar(structure).write_file(run / "POSCAR")
        write_incar(run / "INCAR", incar)
        (run / "KPOINTS").write_text(kpoints)
print("wrote", sorted(str(p) for p in case.glob("*/*/*")))

wrote ['bfd_inputs/VASP_nospin_ibz/np_trans/nscf/INCAR', 'bfd_inputs/VASP_nospin_ibz/np_trans/nscf/KPOINTS', 'bfd_inputs/VASP_nospin_ibz/np_trans/nscf/POSCAR', 'bfd_inputs/VASP_nospin_ibz/np_trans/scf/INCAR', 'bfd_inputs/VASP_nospin_ibz/np_trans/scf/KPOINTS', 'bfd_inputs/VASP_nospin_ibz/np_trans/scf/POSCAR', 'bfd_inputs/VASP_nospin_ibz/pol_orig/nscf/INCAR', 'bfd_inputs/VASP_nospin_ibz/pol_orig/nscf/KPOINTS', 'bfd_inputs/VASP_nospin_ibz/pol_orig/nscf/POSCAR', 'bfd_inputs/VASP_nospin_ibz/pol_orig/scf/INCAR', 'bfd_inputs/VASP_nospin_ibz/pol_orig/scf/KPOINTS', 'bfd_inputs/VASP_nospin_ibz/pol_orig/scf/POSCAR']


### 4. Run the calculations

In each structure's directory, with your `POTCAR` copied into `scf/` and `nscf/`:

```bash
(cd scf && vasp_std > vasp.out)
cp scf/CHGCAR nscf/
(cd nscf && vasp_std > vasp.out)
```

berry-flux-diag reads the NSCF: keep `nscf/POSCAR`, `POTCAR`, `WAVECAR` and `vasprun.xml`.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#vasp-nospin-ibz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="vasp-spin-fbz"></a>
## VASP: spin-polarized, full Brillouin zone

What you will set up: one VASP run per structure; two spin channels, with the moment of each atom set below; symmetry switched off, so the wavefunctions cover every k-point.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [9]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run: ['pol_orig', 'np_trans']


### 2. The magnetic moments

One starting moment per atom, in the order of the structure, in μB. BaTiO₃ is non-magnetic, so every moment is zero here; for an antiferromagnet give opposite signs, e.g. `[-5, 5, 0, ...]`. The total moment is held fixed at zero (`NUPDOWN = 0` / `tot_magnetization = 0`), which suits non-magnetic and antiferromagnetic materials. For a ferromagnet, set it to the total moment instead, or remove it.

In [10]:
magmoms = [0.0] * len(pol)
print("MAGMOM =", " ".join(str(m) for m in magmoms))

MAGMOM = 0.0 0.0 0.0 0.0 0.0


### 3. Write the VASP inputs

Every run needs:
- `ISYM = -1`: no symmetry, so every k-point of the mesh is computed;
- `LWAVE = .TRUE.`: write the `WAVECAR`;
- `NBANDS` fixed, so all runs have the same bands;
- the same `KPOINTS` mesh for every structure.
- `ISPIN = 2`, `MAGMOM`, and `NUPDOWN`.

A `POTCAR` can't be generated here: add your own to each run directory, with the elements in the order of the `POSCAR`.

In [11]:
case = OUT / "VASP_spin_fbz"
incar = {**BASE_INCAR, "ISYM": -1, "LWAVE": True, "NBANDS": 32,
         "ISPIN": 2, "MAGMOM": " ".join(str(m) for m in magmoms), "NUPDOWN": 0}
kpoints = "Automatic mesh\n0\nMonkhorst\n" + " ".join(map(str, VASP_MESH)) + "\n"

for tag, structure in zip(tags, structs):
    run = case / tag
    run.mkdir(parents=True, exist_ok=True)
    Poscar(structure).write_file(run / "POSCAR")
    write_incar(run / "INCAR", incar)
    (run / "KPOINTS").write_text(kpoints)
print("wrote", sorted(str(p) for p in case.glob("*/*")))

wrote ['bfd_inputs/VASP_spin_fbz/np_trans/INCAR', 'bfd_inputs/VASP_spin_fbz/np_trans/KPOINTS', 'bfd_inputs/VASP_spin_fbz/np_trans/POSCAR', 'bfd_inputs/VASP_spin_fbz/pol_orig/INCAR', 'bfd_inputs/VASP_spin_fbz/pol_orig/KPOINTS', 'bfd_inputs/VASP_spin_fbz/pol_orig/POSCAR']


### 4. Run the calculations

In each structure's directory, with your `POTCAR` copied in:

```bash
vasp_std > vasp.out
```

Keep each run's `POSCAR`, `POTCAR`, `WAVECAR` and `vasprun.xml`.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#vasp-spin-fbz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="vasp-spin-ibz"></a>
## VASP: spin-polarized, irreducible Brillouin zone (IBZ)

What you will set up: two VASP runs per structure: an SCF with symmetry on, then an NSCF on the shared IBZ list; two spin channels, with the moment of each atom set below; berry-flux-diag unfolds the IBZ states to the full zone when it reads them.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [12]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run: ['pol_orig', 'np_trans']


### 2. The magnetic moments

One starting moment per atom, in the order of the structure, in μB. BaTiO₃ is non-magnetic, so every moment is zero here; for an antiferromagnet give opposite signs, e.g. `[-5, 5, 0, ...]`. The total moment is held fixed at zero (`NUPDOWN = 0` / `tot_magnetization = 0`), which suits non-magnetic and antiferromagnetic materials. For a ferromagnet, set it to the total moment instead, or remove it.

In [13]:
magmoms = [0.0] * len(pol)
print("MAGMOM =", " ".join(str(m) for m in magmoms))

MAGMOM = 0.0 0.0 0.0 0.0 0.0


### 3. The shared IBZ k-point list

Both runs of a pair must use one identical k-point list, reduced with only the symmetry operations the two structures share. Each code's own automatic reduction would give each structure a different list, so it is computed here. VASP's Monkhorst-Pack mesh is shifted by half a step along even axes. With magnetic moments, only operations that keep every moment are used, so the moments are passed in.

In [14]:
full_mesh = symmetry.monkhorst_pack(VASP_MESH, shift=[1 if n % 2 == 0 else 0 for n in VASP_MESH])
ibz_kpoints, ibz_weights = shared_ibz_kpoints(structs, full_mesh, magmoms)

P4mm (99) and Pm-3m (221): 8 shared operations, 216 -> 18 k-points


### 4. Write the VASP inputs

- **SCF:** symmetry on (no `ISYM`), on the usual mesh.
- **NSCF:** `ICHARG = 11` (read the SCF's charge density), `ISYM = 0` (keep the k-point list as given), the explicit IBZ `KPOINTS`, and `LWAVE = .TRUE.`.
- `NBANDS` is fixed, so the SCF and NSCF, and every structure, have the same bands.
- Both steps: `ISPIN = 2`, `MAGMOM`, and `NUPDOWN`.

A `POTCAR` can't be generated here: add your own to each run directory, with the elements in the order of the `POSCAR`.

In [15]:
case = OUT / "VASP_spin_ibz"
scf_incar = {**BASE_INCAR, "LWAVE": True, "NBANDS": 32,
         "ISPIN": 2, "MAGMOM": " ".join(str(m) for m in magmoms), "NUPDOWN": 0}
# The NSCF reads the SCF's CHGCAR and keeps the k-point list as given. The
# tetrahedron method (ISMEAR = -5) needs a generated mesh, so it uses
# Gaussian smearing instead.
nscf_incar = {**scf_incar, "ICHARG": 11, "ISYM": 0, "ISMEAR": 0, "SIGMA": 0.01,
              "LCHARG": False, "LAECHG": False, "LVTOT": False}
scf_kpoints = "Automatic mesh\n0\nMonkhorst\n" + " ".join(map(str, VASP_MESH)) + "\n"
nscf_kpoints = symmetry.vasp_kpoints_text(ibz_kpoints, ibz_weights)

for tag, structure in zip(tags, structs):
    for step, incar, kpoints in (("scf", scf_incar, scf_kpoints), ("nscf", nscf_incar, nscf_kpoints)):
        run = case / tag / step
        run.mkdir(parents=True, exist_ok=True)
        Poscar(structure).write_file(run / "POSCAR")
        write_incar(run / "INCAR", incar)
        (run / "KPOINTS").write_text(kpoints)
print("wrote", sorted(str(p) for p in case.glob("*/*/*")))

wrote ['bfd_inputs/VASP_spin_ibz/np_trans/nscf/INCAR', 'bfd_inputs/VASP_spin_ibz/np_trans/nscf/KPOINTS', 'bfd_inputs/VASP_spin_ibz/np_trans/nscf/POSCAR', 'bfd_inputs/VASP_spin_ibz/np_trans/scf/INCAR', 'bfd_inputs/VASP_spin_ibz/np_trans/scf/KPOINTS', 'bfd_inputs/VASP_spin_ibz/np_trans/scf/POSCAR', 'bfd_inputs/VASP_spin_ibz/pol_orig/nscf/INCAR', 'bfd_inputs/VASP_spin_ibz/pol_orig/nscf/KPOINTS', 'bfd_inputs/VASP_spin_ibz/pol_orig/nscf/POSCAR', 'bfd_inputs/VASP_spin_ibz/pol_orig/scf/INCAR', 'bfd_inputs/VASP_spin_ibz/pol_orig/scf/KPOINTS', 'bfd_inputs/VASP_spin_ibz/pol_orig/scf/POSCAR']


### 5. Run the calculations

In each structure's directory, with your `POTCAR` copied into `scf/` and `nscf/`:

```bash
(cd scf && vasp_std > vasp.out)
cp scf/CHGCAR nscf/
(cd nscf && vasp_std > vasp.out)
```

berry-flux-diag reads the NSCF: keep `nscf/POSCAR`, `POTCAR`, `WAVECAR` and `vasprun.xml`.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#vasp-spin-ibz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="qe-nospin-fbz"></a>
## Quantum ESPRESSO: non-spin-polarized, full Brillouin zone

What you will set up: one `pw.x` run per structure; no spin polarization; symmetry switched off, so the wavefunctions cover every k-point.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [16]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run:

 ['pol_orig', 'np_trans']


### 2. Write the QE inputs

Every run needs `nosym = .true.` and `noinv = .true.`, so that every k-point of the mesh is computed, a fixed `nbnd`, and the same k-point mesh.

In [17]:
case = OUT / "QE_nospin_fbz"
kpoints_card = "K_POINTS automatic\n  " + " ".join(map(str, QE_MESH + QE_SHIFT)) + "\n"

for tag, structure in zip(tags, structs):
    run = case / tag
    run.mkdir(parents=True, exist_ok=True)
    prefix = f"{MATERIAL}_{tag}"
    write_pw_input(run / f"{prefix}.in", structure, prefix, QE_SETTINGS,
                   calculation="scf", kpoints_card=kpoints_card)
print("wrote", sorted(str(p) for p in case.glob("*/*.in")))

wrote ['bfd_inputs/QE_nospin_fbz/np_trans/BaTiO3_np_trans.in', 'bfd_inputs/QE_nospin_fbz/pol_orig/BaTiO3_pol_orig.in']


### 3. Run the calculations

In each structure's directory:

```bash
pw.x -input BaTiO3_<tag>.in > BaTiO3_<tag>.out
```

Keep `calculations/<prefix>.xml`, the `calculations/<prefix>.save/` folder (with the wavefunctions in HDF5, the default for QE 7) and the `pw.x` output.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#qe-nospin-fbz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="qe-nospin-ibz"></a>
## Quantum ESPRESSO: non-spin-polarized, irreducible Brillouin zone (IBZ)

What you will set up: two `pw.x` runs per structure: an SCF with symmetry on, then an NSCF on the shared IBZ list; no spin polarization; berry-flux-diag unfolds the IBZ states to the full zone when it reads them.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [18]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run:

 ['pol_orig', 'np_trans']


### 2. The shared IBZ k-point list

Both runs of a pair must use one identical k-point list, reduced with only the symmetry operations the two structures share. Each code's own automatic reduction would give each structure a different list, so it is computed here. This is the mesh of `K_POINTS automatic 5 5 5 1 1 1`.

In [19]:
full_mesh = symmetry.monkhorst_pack(QE_MESH, shift=QE_SHIFT)
ibz_kpoints, ibz_weights = shared_ibz_kpoints(structs, full_mesh)

P4mm (99) and Pm-3m (221): 8 shared operations, 125 -> 18 k-points


### 3. Write the QE inputs

- **SCF:** symmetry on, on the usual mesh.
- **NSCF:** `calculation = 'nscf'`, `nosym` and `noinv` (keep the list as given), and the explicit `K_POINTS crystal` list. It has the same `prefix` and `outdir` as the SCF, so it reads the SCF's charge density and overwrites its wavefunctions.
- `nbnd` is fixed, so the SCF and NSCF, and every structure, have the same bands.

In [20]:
case = OUT / "QE_nospin_ibz"
scf_card = "K_POINTS automatic\n  " + " ".join(map(str, QE_MESH + QE_SHIFT)) + "\n"
nscf_card = symmetry.qe_kpoints_text(ibz_kpoints, ibz_weights)

for tag, structure in zip(tags, structs):
    run = case / tag
    run.mkdir(parents=True, exist_ok=True)
    prefix = f"{MATERIAL}_{tag}"
    write_pw_input(run / f"{prefix}_scf.in", structure, prefix, QE_SETTINGS,
                   calculation="scf", kpoints_card=scf_card, use_symmetry=True)
    write_pw_input(run / f"{prefix}_nscf.in", structure, prefix, QE_SETTINGS,
                   calculation="nscf", kpoints_card=nscf_card)
print("wrote", sorted(str(p) for p in case.glob("*/*.in")))

wrote ['bfd_inputs/QE_nospin_ibz/np_trans/BaTiO3_np_trans_nscf.in', 'bfd_inputs/QE_nospin_ibz/np_trans/BaTiO3_np_trans_scf.in', 'bfd_inputs/QE_nospin_ibz/pol_orig/BaTiO3_pol_orig_nscf.in', 'bfd_inputs/QE_nospin_ibz/pol_orig/BaTiO3_pol_orig_scf.in']


### 4. Run the calculations

In each structure's directory, the SCF and then the NSCF:

```bash
pw.x -input BaTiO3_<tag>_scf.in > BaTiO3_<tag>_scf.out
pw.x -input BaTiO3_<tag>_nscf.in > BaTiO3_<tag>_nscf.out
```

berry-flux-diag reads the NSCF: keep `calculations/<prefix>.xml`, `calculations/<prefix>.save/` and the NSCF's `pw.x` output.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#qe-nospin-ibz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="qe-spin-fbz"></a>
## Quantum ESPRESSO: spin-polarized, full Brillouin zone

What you will set up: one `pw.x` run per structure; two spin channels, with the moment of each atom set below; symmetry switched off, so the wavefunctions cover every k-point.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [21]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run: ['pol_orig', 'np_trans']


### 2. The magnetic moments

One starting moment per atom, in the order of the structure, in μB. BaTiO₃ is non-magnetic, so every moment is zero here; for an antiferromagnet give opposite signs, e.g. `[-5, 5, 0, ...]`. The total moment is held fixed at zero (`NUPDOWN = 0` / `tot_magnetization = 0`), which suits non-magnetic and antiferromagnetic materials. For a ferromagnet, set it to the total moment instead, or remove it.

In [22]:
magmoms = [0.0] * len(pol)

# QE takes a starting magnetization per species (between -1 and 1), in the
# order of QE_SETTINGS["species"]. An antiferromagnet needs its up and down
# atoms as two species, e.g. Fe1 and Fe2.
spin_settings = {"nspin": 2, "tot_magnetization": 0,
                 "starting_magnetization(1)": 0}
print(spin_settings)

{'nspin': 2, 'tot_magnetization': 0, 'starting_magnetization(1)': 0}


### 3. Write the QE inputs

Every run needs `nosym = .true.` and `noinv = .true.`, so that every k-point of the mesh is computed, a fixed `nbnd`, and the same k-point mesh. The spin settings go in `&SYSTEM`.

In [23]:
case = OUT / "QE_spin_fbz"
kpoints_card = "K_POINTS automatic\n  " + " ".join(map(str, QE_MESH + QE_SHIFT)) + "\n"

for tag, structure in zip(tags, structs):
    run = case / tag
    run.mkdir(parents=True, exist_ok=True)
    prefix = f"{MATERIAL}_{tag}"
    write_pw_input(run / f"{prefix}.in", structure, prefix, QE_SETTINGS,
                   calculation="scf", kpoints_card=kpoints_card, spin=spin_settings)
print("wrote", sorted(str(p) for p in case.glob("*/*.in")))

wrote ['bfd_inputs/QE_spin_fbz/np_trans/BaTiO3_np_trans.in', 'bfd_inputs/QE_spin_fbz/pol_orig/BaTiO3_pol_orig.in']


### 4. Run the calculations

In each structure's directory:

```bash
pw.x -input BaTiO3_<tag>.in > BaTiO3_<tag>.out
```

Keep `calculations/<prefix>.xml`, the `calculations/<prefix>.save/` folder (with the wavefunctions in HDF5, the default for QE 7) and the `pw.x` output.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#qe-spin-fbz) in the postprocessing notebook.

[Back to contents](#contents)

<a id="qe-spin-ibz"></a>
## Quantum ESPRESSO: spin-polarized, irreducible Brillouin zone (IBZ)

What you will set up: two `pw.x` runs per structure: an SCF with symmetry on, then an NSCF on the shared IBZ list; two spin channels, with the moment of each atom set below; berry-flux-diag unfolds the IBZ states to the full zone when it reads them.

### 1. The polar and non-polar structures

Both must be relaxed already, and list the same atoms in the same order. `preprocess_structs` translates the non-polar structure to minimize the atomic displacements, and adds interpolated images if an atom still moves more than `max_disp` (Å). Each structure in `structs` needs its own DFT run(s).

In [24]:
pol = Structure.from_file("structures/POSCAR_BaTiO3_pol")
non_pol = Structure.from_file("structures/POSCAR_BaTiO3_np")

structs = bfd.preprocess.preprocess_structs(pol, non_pol, max_disp=0.3)["structs"]
tags = image_tags(structs)
print("structures to run:", tags)

structures to run: ['pol_orig', 'np_trans']


### 2. The magnetic moments

One starting moment per atom, in the order of the structure, in μB. BaTiO₃ is non-magnetic, so every moment is zero here; for an antiferromagnet give opposite signs, e.g. `[-5, 5, 0, ...]`. The total moment is held fixed at zero (`NUPDOWN = 0` / `tot_magnetization = 0`), which suits non-magnetic and antiferromagnetic materials. For a ferromagnet, set it to the total moment instead, or remove it.

In [25]:
magmoms = [0.0] * len(pol)

# QE takes a starting magnetization per species (between -1 and 1), in the
# order of QE_SETTINGS["species"]. An antiferromagnet needs its up and down
# atoms as two species, e.g. Fe1 and Fe2.
spin_settings = {"nspin": 2, "tot_magnetization": 0,
                 "starting_magnetization(1)": 0}
print(spin_settings)

{'nspin': 2, 'tot_magnetization': 0, 'starting_magnetization(1)': 0}


### 3. The shared IBZ k-point list

Both runs of a pair must use one identical k-point list, reduced with only the symmetry operations the two structures share. Each code's own automatic reduction would give each structure a different list, so it is computed here. This is the mesh of `K_POINTS automatic 5 5 5 1 1 1`. With magnetic moments, only operations that keep every moment are used, so the moments are passed in.

In [26]:
full_mesh = symmetry.monkhorst_pack(QE_MESH, shift=QE_SHIFT)
ibz_kpoints, ibz_weights = shared_ibz_kpoints(structs, full_mesh, magmoms)

P4mm (99) and Pm-3m (221): 8 shared operations, 125 -> 18 k-points


### 4. Write the QE inputs

- **SCF:** symmetry on, on the usual mesh.
- **NSCF:** `calculation = 'nscf'`, `nosym` and `noinv` (keep the list as given), and the explicit `K_POINTS crystal` list. It has the same `prefix` and `outdir` as the SCF, so it reads the SCF's charge density and overwrites its wavefunctions.
- `nbnd` is fixed, so the SCF and NSCF, and every structure, have the same bands.
- Both steps: the spin settings in `&SYSTEM`.

In [27]:
case = OUT / "QE_spin_ibz"
scf_card = "K_POINTS automatic\n  " + " ".join(map(str, QE_MESH + QE_SHIFT)) + "\n"
nscf_card = symmetry.qe_kpoints_text(ibz_kpoints, ibz_weights)

for tag, structure in zip(tags, structs):
    run = case / tag
    run.mkdir(parents=True, exist_ok=True)
    prefix = f"{MATERIAL}_{tag}"
    write_pw_input(run / f"{prefix}_scf.in", structure, prefix, QE_SETTINGS,
                   calculation="scf", kpoints_card=scf_card, use_symmetry=True, spin=spin_settings)
    write_pw_input(run / f"{prefix}_nscf.in", structure, prefix, QE_SETTINGS,
                   calculation="nscf", kpoints_card=nscf_card, spin=spin_settings)
print("wrote", sorted(str(p) for p in case.glob("*/*.in")))

wrote ['bfd_inputs/QE_spin_ibz/np_trans/BaTiO3_np_trans_nscf.in', 'bfd_inputs/QE_spin_ibz/np_trans/BaTiO3_np_trans_scf.in', 'bfd_inputs/QE_spin_ibz/pol_orig/BaTiO3_pol_orig_nscf.in', 'bfd_inputs/QE_spin_ibz/pol_orig/BaTiO3_pol_orig_scf.in']


### 5. Run the calculations

In each structure's directory, the SCF and then the NSCF:

```bash
pw.x -input BaTiO3_<tag>_scf.in > BaTiO3_<tag>_scf.out
pw.x -input BaTiO3_<tag>_nscf.in > BaTiO3_<tag>_nscf.out
```

berry-flux-diag reads the NSCF: keep `calculations/<prefix>.xml`, `calculations/<prefix>.save/` and the NSCF's `pw.x` output.

**Next:** [compute the polarization](postprocess_DFT_files.ipynb#qe-spin-ibz) in the postprocessing notebook.

[Back to contents](#contents)